# AutoML Tutorial: H2O AutoML
> **Run this notebook in Google Colab or Jupyter to learn about automated machine learning (AutoML) using H2O AutoML.**

## 1. Introduction
Automated Machine Learning (AutoML) simplifies the end-to-end ML workflow by:
- **Feature preprocessing**
- **Model selection**
- **Hyperparameter optimization**
- **Ensembling**

In this tutorial, we'll focus solely on **H2O AutoML**, a scalable AutoML framework supporting both regression and classification.


At the end, you'll complete an exercise applying AutoML.



## 2. Setup & Installation

In [1]:
!pip install --quiet jedi
!pip install --quiet h2o
!pip install --quiet 'thinc<8.3.6'

The system cannot find the file specified.


## Regression on California Housing


In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 3. Regression Example: California Housing

In [3]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import h2o
from h2o.automl import H2OAutoML
from sklearn.metrics import mean_squared_error, r2_score

In [4]:
# Initialize H2O
h2o.init(max_mem_size="2G", nthreads=-1)

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; Java HotSpot(TM) 64-Bit Server VM (build 25.0.2+10-LTS-69, mixed mode, sharing)
  Starting server from C:\Users\binta\miniconda3\envs\ai-bootcamp\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\binta\AppData\Local\Temp\tmpmf0y9sfs
  JVM stdout: C:\Users\binta\AppData\Local\Temp\tmpmf0y9sfs\h2o_binta_started_from_python.out
  JVM stderr: C:\Users\binta\AppData\Local\Temp\tmpmf0y9sfs\h2o_binta_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,05 secs
H2O_cluster_timezone:,Asia/Karachi
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 6 days
H2O_cluster_name:,H2O_from_python_binta_zfjmxl
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.983 Gb
H2O_cluster_total_cores:,0
H2O_cluster_allowed_cores:,0
H2O_cluster_status:,"locked, healthy"


In [5]:
# Load data
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target.rename('target')

In [6]:
# Create H2OFrame
df = h2o.H2OFrame(pd.concat([X, y], axis=1))

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [7]:
# Split into train/test
train, test = df.split_frame(ratios=[0.8], seed=42)

### 3.1 Run H2O AutoML for Regression

In [8]:
aml_reg = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    project_name="california_regression"
)
aml_reg.train(x=X.columns.tolist(), y='target', training_frame=train)

AutoML progress: |
14:58:08.503: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_4_AutoML_1_20260728_145808


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    100                100                         282983                 10           10           10            55            449           220.93

ModelMetricsRegression: gbm
** Reported on train data. **

MSE: 0.07068300661149557
RMSE: 0.2658627589781908
MAE: 0.18120917114794807
RMSLE: 0.08210427216080109
Mean Residual Deviance: 0.07068300661149557

ModelMetricsRegression: gbm
** Reported on cross-validation data. **

MSE: 0.2034847118566311
RMSE: 0.4510927973894408
MAE: 0.29284213433945305
RMSLE: 0.13524205346487878
Mean Residual Deviance: 0.2034847118566311

Cross-Validation Metrics Summary: 
                        mean      sd          cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  --------  ----------  ------------  ------------  ------------  ------------  ------------
aic                     nan       0           nan           nan           nan           nan           nan
loglikelihood           nan       0           nan           nan           nan           nan           nan
mae                     0.292841  0.00338635  0.296661      0.287396      0.293333      0.294019      0.292795
mean_residual_deviance  0.203505  0.00921118  0.217368      0.199415      0.205641      0.202794      0.192307
mse                     0.203505  0.00921118  0.217368      0.199415      0.205641      0.202794      0.192307
r2                      0.8475    0.0055502   0.840433      0.849944      0.845547      0.846212      0.855363
residual_deviance       0.203505  0.00921118  0.217368      0.199415      0.205641      0.202794      0.192307
rmse                    0.451024  0.0101703   0.466228      0.44656       0.453476      0.450326      0.438528
rmsle                   0.135246  0.00145376  0.136438      0.134214      0.136616      0.135689      0.133273

Scoring History: 
     timestamp            duration    number_of_trees    training_rmse        training_mae         training_deviance
---  -------------------  ----------  -----------------  -------------------  -------------------  -------------------
     2026-07-28 14:58:47  3.138 sec   0.0                1.155086044742462    0.9126682923030255   1.3342237707587847
     2026-07-28 14:58:47  3.216 sec   5.0                0.8050718297659779   0.6296027937088264   0.6481406510827397
     2026-07-28 14:58:47  3.281 sec   10.0               0.6022507490332858   0.45921175920543444  0.3627059647111538
     2026-07-28 14:58:47  3.343 sec   15.0               0.4899573038789042   0.36044403364806166  0.24005815962428487
     2026-07-28 14:58:47  3.405 sec   20.0               0.4331452215860772   0.30982096116895946  0.18761478298285192
     2026-07-28 14:58:47  3.459 sec   25.0               0.39683911573298336  0.2769064720926289   0.15748128377573614
     2026-07-28 14:58:47  3.513 sec   30.0               0.374152270539361    0.25627071575137306  0.13998992154975917
     2026-07-28 14:58:47  3.567 sec   35.0               0.3584411213603489   0.24324635480521686  0.12848003748206438
     2026-07-28 14:58:48  3.612 sec   40.0               0.3446381516088021   0.2327461465752868   0.11877545554433169
     2026-07-28 14:58:48  3.656 sec   45.0               0.3338572144190779   0.2247170105589096   0.11146063961966617
---  ---                  ---         ---                ---                  ---                  ---
     2026-07-28 14:58:48  3.745 sec   55.0               0.31870769879870814  0.21496977659685737  0.10157459727356806
     2026-07-28 14:58:48  3.825 sec   60.0               0

In [9]:
# Show leaderboard
df_leader_reg = aml_reg.leaderboard.as_data_frame()
print(df_leader_reg.head())

                                      model_id      rmse       mse       mae  \
0               GBM_4_AutoML_1_20260728_145808  0.451093  0.203485  0.292842   
1               GBM_3_AutoML_1_20260728_145808  0.451954  0.204262  0.297087   
2               GBM_2_AutoML_1_20260728_145808  0.456719  0.208592  0.301624   
3               GBM_1_AutoML_1_20260728_145808  0.458936  0.210623  0.303082   
4  GBM_grid_1_AutoML_1_20260728_145808_model_5  0.461007  0.212527  0.300461   

      rmsle  mean_residual_deviance  
0  0.135242                0.203485  
1  0.136120                0.204262  
2  0.137787                0.208592  
3  0.138777                0.210623  
4  0.138387                0.212527  


c:\Users\binta\miniconda3\envs\ai-bootcamp\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [10]:
# Evaluate on test
perf_reg = aml_reg.leader.model_performance(test)
print(f"H2O Regression R²: {perf_reg.r2():.4f}")
print(f"H2O Regression RMSE: {perf_reg.rmse():.4f}")

H2O Regression R²: 0.8569
H2O Regression RMSE: 0.4346


## 4. Classification Example: Breast Cancer Dataset

In [11]:
from sklearn.datasets import load_breast_cancer

In [12]:
# Load and prepare dataset
data_cls = load_breast_cancer(as_frame=True)
Xc = data_cls.data
yc = data_cls.target.rename('target')

In [13]:
df_cls = h2o.H2OFrame(pd.concat([Xc, yc], axis=1))
train_cls, test_cls = df_cls.split_frame(ratios=[0.7], seed=42)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


### 4.1 Run H2O AutoML for Classification

In [14]:
aml_cls = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    balance_classes=True,
    project_name="breast_cancer_classification"
)
aml_cls.train(x=Xc.columns.tolist(), y='target', training_frame=train_cls)

AutoML progress: |
15:04:04.143: AutoML: XGBoost is not available; skipping it.
15:04:04.160: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.


15:04:04.457: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.


15:04:04.935: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

██
15:04:05.715: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical b

key,value
Stacking strategy,cross_validation
Number of base models (used / total),10/28
# GBM base models (used / total),8/25
# DRF base models (used / total),1/2
# GLM base models (used / total),1/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None
Custom metalearner hyperparameters,None


In [15]:
# Show leaderboard
df_leader_cls = aml_cls.leaderboard.as_data_frame()
print(df_leader_cls.head())

                                            model_id      rmse       mse  \
0  StackedEnsemble_AllModels_1_AutoML_2_20260728_...  0.181630  0.032989   
1  StackedEnsemble_BestOfFamily_1_AutoML_2_202607...  0.182539  0.033320   
2       GBM_grid_1_AutoML_2_20260728_150404_model_25  0.185965  0.034583   
3       GBM_grid_1_AutoML_2_20260728_150404_model_14  0.186716  0.034863   
4       GBM_grid_1_AutoML_2_20260728_150404_model_23  0.188528  0.035543   

        mae     rmsle  mean_residual_deviance  
0  0.097113  0.131007                0.032989  
1  0.104637  0.131654                0.033320  
2  0.096905  0.131893                0.034583  
3  0.089201  0.134120                0.034863  
4  0.090711  0.133354                0.035543  


c:\Users\binta\miniconda3\envs\ai-bootcamp\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [16]:
# Evaluate on test
perf_cls = aml_cls.leader.model_performance(test_cls)
print(f"H2O Classification: {perf_cls}")

H2O Classification: ModelMetricsRegressionGLM: stackedensemble
** Reported on test data. **

MSE: 0.03556103996032552
RMSE: 0.18857635047991972
MAE: 0.09881519448380302
RMSLE: 0.12886513962124194
Mean Residual Deviance: 0.03556103996032552
R^2: 0.8421303082383419
Null degrees of freedom: 177
Residual degrees of freedom: 167
Null deviance: 40.43221198186829
Residual deviance: 6.329865112937943
AIC: -64.75570516846703


MSE (Mean Squared Error):
- Measures the average squared difference between predicted and actual values. Lower values indicate better performance.


RMSE (Root Mean Squared Error):
- The square root of MSE, providing an error metric in the same units as the target variable. Lower values are better.


MAE (Mean Absolute Error):
-Measures the average absolute difference between predicted and actual values. Lower values indicate better performance.


RMSLE (Root Mean Squared Logarithmic Error):
- Similar to RMSE but uses the logarithm of the values, making it more robust to outliers. Lower values are better.


Mean Residual Deviance:
- Measures the goodness of fit of the model. Lower values indicate a better fit.


R² (Coefficient of Determination):
- Represents the proportion of variance explained by the model. Values closer to 1 indicate better performance.


Null Degrees of Freedom:
- The number of observations minus 1.


Residual Degrees of Freedom:
- The number of observations minus the number of parameters estimated.


Null Deviance:
- The deviance of the null model (model with no predictors).


Residual Deviance:
- The deviance of the fitted model. Lower values indicate a better fit.


AIC (Akaike Information Criterion):
- A measure of model quality, balancing goodness of fit and model complexity. Lower values indicate a better model.

## 5. Interpreting Results
- **Leaderboard** displays model ranking by default metric.
- Use `model_performance` to compute custom metrics (RMSE, R², AUC, accuracy, etc.).
- Top models are automatically ensembled by H2O's Stacked Ensemble.

## 6. AutoML Best Practices
- **Set time and model limits** (`max_runtime_secs`, `max_models`) to control cost and runtime.
- **Use cross-validation** (`nfolds`) for robust performance estimates.
- **Balance classes** for imbalanced classification tasks.
- **Review variable importance** on the leader model: `aml.leader.varimp()`.
- **Save and deploy** the best model: `h2o.save_model(aml.leader, path='best_model')`.

## 7. Exercise: Custom Dataset AutoML

**Task:** Apply H2O AutoML to a custom dataset.

1. Load any tabular dataset (CSV or from `sklearn.datasets`).
2. Decide whether it's a regression or classification task.
3. Initialize H2O and convert to `H2OFrame`.
4. Split into appropriate train/test ratios.
5. Run `H2OAutoML` with:
   - `max_runtime_secs=300`
   - `max_models=15`
   - `nfolds=5`
   - `balance_classes=True` (if classification)
6. Display the leaderboard and evaluate on the test set using relevant metrics.
7. Save the leaderboard to a pandas DataFrame and export it as `leaderboard.csv`

In [18]:
# Exercise Solution: H2O AutoML on a Custom Dataset

import h2o
from h2o.automl import H2OAutoML
from sklearn.datasets import load_wine

# Initialize H2O
h2o.init()

# Load a custom dataset (Wine Classification)
wine = load_wine(as_frame=True)
df = wine.frame

# Convert target to categorical (classification)
df["target"] = df["target"].astype("category")

# Convert to H2OFrame
hf = h2o.H2OFrame(df)

# Define target and predictors
y = "target"
x = [col for col in hf.columns if col != y]

# Ensure target is categorical
hf[y] = hf[y].asfactor()

# Split data
train, test = hf.split_frame(ratios=[0.8], seed=42)

# Run AutoML
aml = H2OAutoML(
    max_runtime_secs=300,
    max_models=15,
    seed=42,
    nfolds=5,
    balance_classes=True
)

aml.train(x=x, y=y, training_frame=train)

# Display leaderboard
leaderboard = aml.leaderboard
print(leaderboard)

# Evaluate on test data
performance = aml.leader.model_performance(test)
print(performance)

# Save leaderboard
leaderboard_df = leaderboard.as_data_frame()
leaderboard_df.to_csv("leaderboard.csv", index=False)

print("Leaderboard saved as leaderboard.csv")

# Save best model
model_path = h2o.save_model(
    model=aml.leader,
    path="best_model",
    force=True
)

print("Best model saved at:", model_path)

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,12 mins 48 secs
H2O_cluster_timezone:,Asia/Karachi
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 6 days
H2O_cluster_name:,H2O_from_python_binta_zfjmxl
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.756 Gb
H2O_cluster_total_cores:,12
H2O_cluster_allowed_cores:,12
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |█
15:10:49.696: AutoML: XGBoost is not available; skipping it.
15:10:50.488: _min_rows param, The dataset size is too small to split for min_rows=100.0: must have at least 200.0 (weighted) rows, but have only 146.0.

██████████████████████████████████████████████████████████████| (done) 100%
model_id                                                   mean_per_class_error    logloss      rmse        mse
XRT_1_AutoML_4_20260728_151049                                        0.0114943  0.136519   0.169509  0.0287331
GBM_grid_1_AutoML_4_20260728_151049_model_1                           0.0114943  0.0591323  0.123506  0.0152536
GBM_3_AutoML_4_20260728_151049                                        0.0114943  0.0360966  0.10548   0.011126
GBM_2_AutoML_4_20260728_151049                                        0.0114943  0.0386741  0.110262  0.0121577
DeepLearning_grid_1_AutoML_4_202607

c:\Users\binta\miniconda3\envs\ai-bootcamp\lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


Leaderboard saved as leaderboard.csv
Best model saved at: C:\Users\binta\OneDrive\Desktop\bootcamp\best_model\XRT_1_AutoML_4_20260728_151049


https://github.com/zainabf07/AI-bootcamp
refer to branch 'lab-15'